In [1]:
# ============================================================
# Notebook 02: Volatility-Sorted Technical Anomaly
# Direction 1: Commodity Futures version of Han, Yang, and Zhou
# Execution timing: Version A delayed execution, audited delayed execution with fixed vol-bucket and HML alignment
#
# Signal date s:
#   close[s], MA[s], RV60[s], vol_bucket[s] are observed after close[s].
# Execution date t = next trading date:
#   position[t] = signal_position[s]
#   strategy_return[t] = position[t] * fwd_ret_1d[t]
#
# This avoids assuming that a signal formed with close[s] can trade the same close[s].
# ============================================================

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

warnings.filterwarnings("ignore")

# ============================================================
# 1. Config and paths
# ============================================================
RUN_TIME = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

INPUT_PARQUET = Path(
    "/home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.parquet"
)
INPUT_CSV = Path(
    "/home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.csv"
)

OUTPUT_DIR = Path(
    "/home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly"
)
TABLE_DIR = OUTPUT_DIR / "tables"
RETURNS_DIR = OUTPUT_DIR / "strategy_returns"
FEATURE_DIR = OUTPUT_DIR / "features"

for p in [OUTPUT_DIR, TABLE_DIR, RETURNS_DIR, FEATURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

FEATURE_PANEL_PARQUET = FEATURE_DIR / "notebook02_feature_panel.parquet"
FEATURE_PANEL_CSV = FEATURE_DIR / "notebook02_feature_panel.csv"

SYMBOL_RETURNS_PARQUET = RETURNS_DIR / "notebook02_symbol_ma_strategy_returns.parquet"
SYMBOL_RETURNS_CSV = RETURNS_DIR / "notebook02_symbol_ma_strategy_returns.csv"

BUCKET_RETURNS_PARQUET = RETURNS_DIR / "notebook02_bucket_daily_returns.parquet"
BUCKET_RETURNS_CSV = RETURNS_DIR / "notebook02_bucket_daily_returns.csv"

HML_RETURNS_PARQUET = RETURNS_DIR / "notebook02_high_minus_low_daily_returns.parquet"
HML_RETURNS_CSV = RETURNS_DIR / "notebook02_high_minus_low_daily_returns.csv"

PERF_TABLE_PATH = TABLE_DIR / "notebook02_bucket_performance_summary.csv"
HML_SUMMARY_PATH = TABLE_DIR / "notebook02_high_minus_low_summary.csv"
MONOTONIC_FLAGS_PATH = TABLE_DIR / "notebook02_monotonic_flags.csv"
SAMPLE_COVERAGE_PATH = TABLE_DIR / "notebook02_sample_coverage.csv"
RUN_CONFIG_PATH = TABLE_DIR / "notebook02_run_config.csv"

MA_WINDOWS = [10, 20, 60, 120]
RV_WINDOW = 60
MIN_CROSS_SECTION_SYMBOLS = 9
COST_BPS = 5.0
TRADING_DAYS = 252

# delayed execution Version 
EXECUTION_MODE = "delayed_one_trading_day"

print("RUN_TIME:", RUN_TIME)
print("INPUT_PARQUET:", INPUT_PARQUET)
print("INPUT_CSV:", INPUT_CSV)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MA_WINDOWS:", MA_WINDOWS)
print("RV_WINDOW:", RV_WINDOW)
print("COST_BPS:", COST_BPS)
print("EXECUTION_MODE:", EXECUTION_MODE)


# ============================================================
# 2. Helpers
# ============================================================
def read_panel() -> pd.DataFrame:
    if INPUT_PARQUET.exists():
        print("Loading parquet:", INPUT_PARQUET)
        return pd.read_parquet(INPUT_PARQUET)
    if INPUT_CSV.exists():
        print("Loading csv:", INPUT_CSV)
        return pd.read_csv(INPUT_CSV, parse_dates=["date", "trading_day"])
    raise FileNotFoundError(
        "Notebook 01 clean panel not found. Checked:\n"
        f"{INPUT_PARQUET}\n{INPUT_CSV}"
    )


def safe_to_parquet(df: pd.DataFrame, path: Path) -> bool:
    try:
        df.to_parquet(path, index=False)
        print("Saved parquet:", path)
        return True
    except Exception as e:
        print("Parquet save failed:", path)
        print("Reason:", repr(e))
        return False


def parse_bool_flag(s: pd.Series, name: str) -> pd.Series:
    """
    Convert Notebook 01 tradability flags to strict booleans.

    This avoids pandas treating non-empty strings such as "False" or "0" as True.
    """
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)

    if pd.api.types.is_numeric_dtype(s):
        valid = s.dropna()
        bad = ~valid.isin([0, 1])
        assert not bad.any(), (
            f"Invalid numeric boolean flag values in {name}: "
            f"{sorted(valid[bad].unique())[:10]}"
        )
        return s.fillna(0).astype(int).astype(bool)

    normalized = s.astype("string").str.strip().str.lower()
    normalized = normalized.mask(normalized == "")
    mapped = normalized.map({
        "true": True, "false": False,
        "1": True, "0": False,
        "yes": True, "no": False,
        "y": True, "n": False,
    })
    bad = mapped.isna() & normalized.notna()
    assert not bad.any(), (
        f"Invalid boolean flag values in {name}: "
        f"{sorted(s[bad].dropna().astype(str).unique())[:10]}"
    )
    return mapped.fillna(False).astype(bool)


def assign_daily_tertile_buckets(df: pd.DataFrame, value_col: str = "rv_60_signal") -> pd.Series:
    """
    Assign Low / Mid / High buckets within each date using only signal-date information.

    Important implementation detail:
    Do not reset or reorder indices. The returned Series uses the original row index,
    so assignment back to feature_panel is index-safe.
    """
    out = pd.Series(pd.NA, index=df.index, dtype="object")

    for _, idx in df.groupby("date", sort=True).groups.items():
        values = df.loc[idx, value_col]
        valid = values.notna()
        n = int(valid.sum())

        if n < MIN_CROSS_SECTION_SYMBOLS:
            continue

        ranks = values.loc[valid].rank(method="first", ascending=True)
        pct = (ranks - 1) / n

        out.loc[pct[pct < 1 / 3].index] = "Low"
        out.loc[pct[(pct >= 1 / 3) & (pct < 2 / 3)].index] = "Mid"
        out.loc[pct[pct >= 2 / 3].index] = "High"

    return out


def max_drawdown(return_series: pd.Series) -> float:
    r = return_series.dropna().astype(float)
    if len(r) == 0:
        return np.nan
    wealth = (1.0 + r).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1.0
    return float(dd.min())


def perf_stats(df: pd.DataFrame, group_cols: list, return_col: str) -> pd.DataFrame:
    rows = []
    for keys, x in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        r = x[return_col].dropna().astype(float)
        n = len(r)
        mean_daily = r.mean() if n > 0 else np.nan
        std_daily = r.std(ddof=1) if n > 1 else np.nan
        ann_return = mean_daily * TRADING_DAYS if pd.notna(mean_daily) else np.nan
        ann_vol = std_daily * np.sqrt(TRADING_DAYS) if pd.notna(std_daily) else np.nan
        sharpe = ann_return / ann_vol if pd.notna(ann_vol) and ann_vol != 0 else np.nan
        t_stat = mean_daily / (std_daily / np.sqrt(n)) if n > 1 and pd.notna(std_daily) and std_daily != 0 else np.nan
        win_rate = float((r > 0).mean()) if n > 0 else np.nan
        mdd = max_drawdown(r)

        row = {col: val for col, val in zip(group_cols, keys)}
        # Avoid overwriting a grouped column named "return_col". This matters for
        # HML summaries, where return_col identifies the source strategy metric and
        # the evaluated series is always hml_return.
        if "return_col" not in group_cols:
            row["return_col"] = return_col
        else:
            row["evaluated_return_col"] = return_col

        row.update({
            "n_obs": n,
            "mean_daily": mean_daily,
            "ann_return": ann_return,
            "ann_vol": ann_vol,
            "sharpe": sharpe,
            "t_stat": t_stat,
            "win_rate": win_rate,
            "max_drawdown": mdd,
        })
        rows.append(row)
    return pd.DataFrame(rows)


def monotonic_flag(low: float, mid: float, high: float, hml_t: float) -> str:
    if any(pd.isna(v) for v in [low, mid, high]):
        return "INSUFFICIENT"
    if low < mid < high and pd.notna(hml_t) and hml_t > 1.96:
        return "STRONG"
    if high > low:
        return "PARTIAL"
    if high < low:
        return "REVERSED"
    return "NO_PATTERN"


# ============================================================
# 3. Load and validate clean panel
# ============================================================
panel = read_panel()
panel = panel.copy()
panel["date"] = pd.to_datetime(panel["date"])
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)

print("panel shape:", panel.shape)
print("symbols:", panel["symbol"].nunique())
print("date range:", panel["date"].min().date(), "->", panel["date"].max().date())
display(panel.head())

required_cols = [
    "date", "symbol", "sector", "close", "ret_1d", "fwd_ret_1d", "fwd_log_ret_1d",
    "fwd_tradable_return_flag", "tradable_return_flag",
]
missing = [c for c in required_cols if c not in panel.columns]
assert not missing, f"Missing required columns from Notebook 01: {missing}"
assert "contract_id" not in panel.columns, "contract_id should not exist in clean panel."
assert not panel.duplicated(["symbol", "date"]).any(), "Duplicate symbol-date rows found."

# Normalize tradability flags once, before any sample filtering.
for flag_col in ["tradable_return_flag", "fwd_tradable_return_flag"]:
    panel[flag_col] = parse_bool_flag(panel[flag_col], flag_col)
    assert pd.api.types.is_bool_dtype(panel[flag_col]), f"{flag_col} was not converted to bool."


# ============================================================
# 4. Feature engineering: signal-date RV, MA, and vol buckets
# ============================================================
feature_panel = panel[[
    "date", "symbol", "sector", "close", "ret_1d", "fwd_ret_1d", "fwd_log_ret_1d",
    "tradable_return_flag", "fwd_tradable_return_flag",
]].copy()

# Signal-date features. These are known after close[date].
feature_panel["rv_60_signal"] = (
    feature_panel.groupby("symbol", group_keys=False)["ret_1d"]
    .apply(lambda s: s.rolling(RV_WINDOW, min_periods=RV_WINDOW).std())
)

for w in MA_WINDOWS:
    feature_panel[f"ma_{w}_signal"] = (
        feature_panel.groupby("symbol", group_keys=False)["close"]
        .apply(lambda s: s.rolling(w, min_periods=w).mean())
    )

# Signal-date vol bucket, assigned cross-sectionally by RV60 on the signal date.
# The returned Series is indexed by the original feature_panel row index.
feature_panel["vol_bucket_signal"] = assign_daily_tertile_buckets(feature_panel, "rv_60_signal")

# Hard check: no bucket may exist without a valid RV signal.
bad_bucket_without_rv = (
    feature_panel["vol_bucket_signal"].notna()
    & feature_panel["rv_60_signal"].isna()
)
assert not bad_bucket_without_rv.any(), "Vol bucket assigned to rows without valid RV60."

bucket_quality = (
    feature_panel.groupby("date")
    .agg(
        n_rv60_signal=("rv_60_signal", lambda s: s.notna().sum()),
        n_vol_bucket_signal=("vol_bucket_signal", lambda s: s.notna().sum()),
    )
    .reset_index()
)

bad_bucket_count = bucket_quality[
    ((bucket_quality["n_rv60_signal"] < MIN_CROSS_SECTION_SYMBOLS) & (bucket_quality["n_vol_bucket_signal"] != 0))
    | ((bucket_quality["n_rv60_signal"] >= MIN_CROSS_SECTION_SYMBOLS) & (bucket_quality["n_vol_bucket_signal"] != bucket_quality["n_rv60_signal"]))
]
assert len(bad_bucket_count) == 0, "Vol bucket daily count mismatch."

# Delayed execution: execution row t uses signal information from previous trading row t-1.
feature_panel["signal_date"] = feature_panel.groupby("symbol")["date"].shift(1)
feature_panel["exec_vol_bucket"] = feature_panel.groupby("symbol")["vol_bucket_signal"].shift(1)
feature_panel["exec_rv_60"] = feature_panel.groupby("symbol")["rv_60_signal"].shift(1)

for w in MA_WINDOWS:
    feature_panel[f"exec_ma_{w}"] = feature_panel.groupby("symbol")[f"ma_{w}_signal"].shift(1)
    feature_panel[f"signal_close_{w}"] = feature_panel.groupby("symbol")["close"].shift(1)

# Alignment checks for delayed execution.
_expected_signal_date = feature_panel.groupby("symbol")["date"].shift(1)
assert (
    feature_panel["signal_date"].fillna(pd.Timestamp("1900-01-01"))
    == _expected_signal_date.fillna(pd.Timestamp("1900-01-01"))
).all(), "signal_date is not previous trading date by symbol."

_expected_exec_bucket = feature_panel.groupby("symbol")["vol_bucket_signal"].shift(1)
assert (
    feature_panel["exec_vol_bucket"].fillna("__MISSING__")
    == _expected_exec_bucket.fillna("__MISSING__")
).all(), "exec_vol_bucket is not previous trading date vol bucket by symbol."

_expected_exec_rv = feature_panel.groupby("symbol")["rv_60_signal"].shift(1)
_rv_diff = (feature_panel["exec_rv_60"] - _expected_exec_rv).abs().max(skipna=True)
assert pd.isna(_rv_diff) or _rv_diff < 1e-12, "exec_rv_60 alignment mismatch."

assert not (
    feature_panel["exec_vol_bucket"].notna()
    & feature_panel["exec_rv_60"].isna()
).any(), "exec_vol_bucket exists while exec_rv_60 is missing."

for w in MA_WINDOWS:
    _expected_ma = feature_panel.groupby("symbol")[f"ma_{w}_signal"].shift(1)
    _expected_close = feature_panel.groupby("symbol")["close"].shift(1)
    _ma_diff = (feature_panel[f"exec_ma_{w}"] - _expected_ma).abs().max(skipna=True)
    _close_diff = (feature_panel[f"signal_close_{w}"] - _expected_close).abs().max(skipna=True)
    assert pd.isna(_ma_diff) or _ma_diff < 1e-12, f"exec_ma_{w} alignment mismatch."
    assert pd.isna(_close_diff) or _close_diff < 1e-12, f"signal_close_{w} alignment mismatch."

sample_coverage = (
    feature_panel.groupby("date")
    .agg(
        n_symbols_total=("symbol", "nunique"),
        n_rv60_signal=("rv_60_signal", lambda x: x.notna().sum()),
        n_vol_bucket_signal=("vol_bucket_signal", lambda x: x.notna().sum()),
        n_exec_vol_bucket=("exec_vol_bucket", lambda x: x.notna().sum()),
    )
    .reset_index()
)

sample_coverage.to_csv(SAMPLE_COVERAGE_PATH, index=False, encoding="utf-8-sig")
print("Saved:", SAMPLE_COVERAGE_PATH)
display(sample_coverage.head())
display(feature_panel.head(80))

# Save feature panel for diagnostics.
feature_panel.to_csv(FEATURE_PANEL_CSV, index=False, encoding="utf-8-sig")
safe_to_parquet(feature_panel, FEATURE_PANEL_PARQUET)
print("Saved CSV:", FEATURE_PANEL_CSV)


# ============================================================
# 5. Build delayed MA strategy returns
# ============================================================
records = []
base_cols = [
    "date", "signal_date", "symbol", "sector", "close", "fwd_ret_1d",
    "fwd_log_ret_1d", "fwd_tradable_return_flag",
    "exec_vol_bucket", "exec_rv_60",
]

for w in MA_WINDOWS:
    tmp = feature_panel[base_cols + [f"exec_ma_{w}", f"signal_close_{w}"]].copy()
    tmp = tmp.rename(columns={
        "exec_vol_bucket": "vol_bucket",
        "exec_rv_60": "rv_60",
        f"exec_ma_{w}": "ma_value",
        f"signal_close_{w}": "signal_close",
    })
    tmp["ma_window"] = w

    signal_known = tmp["signal_close"].notna() & tmp["ma_value"].notna()
    above_ma = tmp["signal_close"] > tmp["ma_value"]

    for version in ["long_flat", "long_short"]:
        x = tmp.copy()
        x["version"] = version

        if version == "long_flat":
            x["position"] = np.where(signal_known, np.where(above_ma, 1.0, 0.0), np.nan)
        else:
            x["position"] = np.where(signal_known, np.where(above_ma, 1.0, -1.0), np.nan)

        # Trading window starts on date t and earns fwd_ret_1d[t].
        # Position was decided from previous trading day's signal.
        x["valid_strategy_row"] = (
            x["fwd_tradable_return_flag"]
            & x["position"].notna()
            & x["vol_bucket"].notna()
            & x["signal_date"].notna()
        )

        x = x[x["valid_strategy_row"]].copy()
        x = x.sort_values(["symbol", "date"])

        prev_position = x.groupby("symbol")["position"].shift(1).fillna(0.0)
        x["turnover"] = (x["position"] - prev_position).abs()
        x["cost_return"] = x["turnover"] * COST_BPS / 10000.0

        x["passive_return"] = x["fwd_ret_1d"]
        x["gross_strategy_return"] = x["position"] * x["fwd_ret_1d"]
        x["net_strategy_return"] = x["gross_strategy_return"] - x["cost_return"]

        x["gross_map_vs_passive"] = x["gross_strategy_return"] - x["passive_return"]
        x["net_map_vs_passive"] = x["net_strategy_return"] - x["passive_return"]
        x["net_map_vs_zero"] = x["net_strategy_return"]

        records.append(x)

assert any(len(r) > 0 for r in records), (
    "No valid symbol strategy records generated. Check signal availability, "
    "tradability flags, and vol-bucket coverage."
)
symbol_strategy_returns = pd.concat(records, axis=0, ignore_index=True)
symbol_strategy_returns = symbol_strategy_returns.sort_values(
    ["date", "symbol", "ma_window", "version"]
).reset_index(drop=True)

print("symbol_strategy_returns shape:", symbol_strategy_returns.shape)
display(symbol_strategy_returns.head())
display(symbol_strategy_returns.tail())

symbol_strategy_returns.to_csv(SYMBOL_RETURNS_CSV, index=False, encoding="utf-8-sig")
safe_to_parquet(symbol_strategy_returns, SYMBOL_RETURNS_PARQUET)
print("Saved CSV:", SYMBOL_RETURNS_CSV)


# ============================================================
# 6. Equal-weight bucket returns
# ============================================================
return_cols = [
    "passive_return", "gross_strategy_return", "net_strategy_return",
    "gross_map_vs_passive", "net_map_vs_passive", "net_map_vs_zero",
]

bucket_daily_returns = (
    symbol_strategy_returns
    .groupby(["date", "ma_window", "version", "vol_bucket"], dropna=False)
    .agg(
        n_symbols=("symbol", "nunique"),
        avg_turnover=("turnover", "mean"),
        avg_cost_return=("cost_return", "mean"),
        **{col: (col, "mean") for col in return_cols},
    )
    .reset_index()
)

bucket_daily_returns = bucket_daily_returns.sort_values(
    ["date", "ma_window", "version", "vol_bucket"]
).reset_index(drop=True)

print("bucket_daily_returns shape:", bucket_daily_returns.shape)
display(bucket_daily_returns.head())

bucket_daily_returns.to_csv(BUCKET_RETURNS_CSV, index=False, encoding="utf-8-sig")
safe_to_parquet(bucket_daily_returns, BUCKET_RETURNS_PARQUET)
print("Saved CSV:", BUCKET_RETURNS_CSV)


# ============================================================
# 7. Bucket performance summary
# ============================================================
perf_frames = []
for col in return_cols:
    perf_frames.append(perf_stats(
        bucket_daily_returns,
        group_cols=["ma_window", "version", "vol_bucket"],
        return_col=col,
    ))

bucket_perf_summary = pd.concat(perf_frames, axis=0, ignore_index=True)
bucket_perf_summary = bucket_perf_summary.sort_values(
    ["return_col", "ma_window", "version", "vol_bucket"]
).reset_index(drop=True)

bucket_perf_summary.to_csv(PERF_TABLE_PATH, index=False, encoding="utf-8-sig")
print("Saved:", PERF_TABLE_PATH)
display(bucket_perf_summary.head(30))


# ============================================================
# 8. High-minus-Low daily returns and summary
# ============================================================
hml_records = []

for col in return_cols:
    wide = bucket_daily_returns.pivot_table(
        index=["date", "ma_window", "version"],
        columns="vol_bucket",
        values=col,
        aggfunc="first",
    ).reset_index()

    if "High" not in wide.columns or "Low" not in wide.columns:
        continue

    wide["hml_return"] = wide["High"] - wide["Low"]
    wide["return_col"] = col
    hml_records.append(wide[["date", "ma_window", "version", "return_col", "hml_return"]])

assert hml_records, "No HML records generated. Check that both High and Low buckets exist."
hml_daily_returns = pd.concat(hml_records, axis=0, ignore_index=True)
hml_daily_returns = hml_daily_returns[hml_daily_returns["hml_return"].notna()].copy()
hml_daily_returns = hml_daily_returns.sort_values(
    ["date", "ma_window", "version", "return_col"]
).reset_index(drop=True)

assert hml_daily_returns["hml_return"].notna().all(), "HML return contains NaN after filtering."

hml_daily_returns.to_csv(HML_RETURNS_CSV, index=False, encoding="utf-8-sig")
safe_to_parquet(hml_daily_returns, HML_RETURNS_PARQUET)
print("Saved CSV:", HML_RETURNS_CSV)
display(hml_daily_returns.head())

hml_summary = perf_stats(
    hml_daily_returns,
    group_cols=["ma_window", "version", "return_col"],
    return_col="hml_return",
)
hml_summary = hml_summary.rename(columns={
    "mean_daily": "hml_mean_daily",
    "ann_return": "hml_ann_return",
    "ann_vol": "hml_ann_vol",
    "sharpe": "hml_sharpe",
    "t_stat": "hml_t_stat",
    "win_rate": "hml_win_rate",
    "max_drawdown": "hml_max_drawdown",
})

expected_return_cols = set(return_cols)
actual_hml_return_cols = set(hml_summary["return_col"].dropna().unique())
assert actual_hml_return_cols == expected_return_cols, (
    f"HML summary return_col mismatch. Expected {expected_return_cols}, got {actual_hml_return_cols}"
)

hml_summary.to_csv(HML_SUMMARY_PATH, index=False, encoding="utf-8-sig")
print("Saved:", HML_SUMMARY_PATH)
display(hml_summary)


# ============================================================
# 9. Monotonic flags
# ============================================================
main_return_col = "net_strategy_return"
main_perf = bucket_perf_summary[bucket_perf_summary["return_col"] == main_return_col].copy()
main_wide = main_perf.pivot_table(
    index=["ma_window", "version"],
    columns="vol_bucket",
    values="mean_daily",
    aggfunc="first",
).reset_index()

main_hml = hml_summary[hml_summary["return_col"] == main_return_col][
    ["ma_window", "version", "hml_mean_daily", "hml_t_stat", "hml_sharpe", "n_obs"]
].copy()

monotonic_flags = main_wide.merge(main_hml, on=["ma_window", "version"], how="left")
assert monotonic_flags["hml_t_stat"].notna().all(), (
    "Monotonic flags failed to merge HML statistics for net_strategy_return."
)

for b in ["Low", "Mid", "High"]:
    if b not in monotonic_flags.columns:
        monotonic_flags[b] = np.nan

monotonic_flags["monotonic_flag"] = monotonic_flags.apply(
    lambda r: monotonic_flag(r.get("Low"), r.get("Mid"), r.get("High"), r.get("hml_t_stat")),
    axis=1,
)

monotonic_flags = monotonic_flags.sort_values(["ma_window", "version"]).reset_index(drop=True)
monotonic_flags.to_csv(MONOTONIC_FLAGS_PATH, index=False, encoding="utf-8-sig")
print("Saved:", MONOTONIC_FLAGS_PATH)
display(monotonic_flags)


# ============================================================
# 10. Run config
# ============================================================
run_config = pd.DataFrame([
    {"key": "run_time", "value": RUN_TIME},
    {"key": "input_parquet", "value": str(INPUT_PARQUET)},
    {"key": "input_csv", "value": str(INPUT_CSV)},
    {"key": "output_dir", "value": str(OUTPUT_DIR)},
    {"key": "execution_mode", "value": EXECUTION_MODE},
    {"key": "execution_note", "value": "Version A: position on date t equals signal position from previous trading day."},
    {"key": "return_window", "value": "fwd_ret_1d[t] = close[t+1] / close[t] - 1"},
    {"key": "rv_window", "value": RV_WINDOW},
    {"key": "ma_windows", "value": ",".join(map(str, MA_WINDOWS))},
    {"key": "min_cross_section_symbols", "value": MIN_CROSS_SECTION_SYMBOLS},
    {"key": "cost_bps_per_turnover_unit", "value": COST_BPS},
    {"key": "n_symbol_strategy_rows", "value": len(symbol_strategy_returns)},
    {"key": "n_bucket_daily_rows", "value": len(bucket_daily_returns)},
    {"key": "n_hml_daily_rows", "value": len(hml_daily_returns)},
])

run_config.to_csv(RUN_CONFIG_PATH, index=False, encoding="utf-8-sig")
print("Saved:", RUN_CONFIG_PATH)
display(run_config)


# ============================================================
# 11. Final hard self-check
# ============================================================
print("Notebook 02 final hard self-check")
print("=" * 80)

assert len(symbol_strategy_returns) > 0, "symbol_strategy_returns is empty."
assert len(bucket_daily_returns) > 0, "bucket_daily_returns is empty."
assert len(hml_daily_returns) > 0, "hml_daily_returns is empty."

# No duplicate keys in core outputs.
assert not symbol_strategy_returns.duplicated(["date", "symbol", "ma_window", "version"]).any(), (
    "Duplicate symbol strategy rows."
)
assert not bucket_daily_returns.duplicated(["date", "ma_window", "version", "vol_bucket"]).any(), (
    "Duplicate bucket daily rows."
)
assert not hml_daily_returns.duplicated(["date", "ma_window", "version", "return_col"]).any(), (
    "Duplicate HML daily rows."
)

# Position legality.
lf_pos = set(symbol_strategy_returns.loc[
    symbol_strategy_returns["version"] == "long_flat", "position"
].dropna().unique())
ls_pos = set(symbol_strategy_returns.loc[
    symbol_strategy_returns["version"] == "long_short", "position"
].dropna().unique())
assert lf_pos.issubset({0.0, 1.0}), f"Invalid long_flat positions: {lf_pos}"
assert ls_pos.issubset({-1.0, 1.0}), f"Invalid long_short positions: {ls_pos}"

# Formula checks.
calc_gross = symbol_strategy_returns["position"] * symbol_strategy_returns["fwd_ret_1d"]
calc_cost = symbol_strategy_returns["turnover"] * COST_BPS / 10000.0
calc_net = symbol_strategy_returns["gross_strategy_return"] - symbol_strategy_returns["cost_return"]
calc_map = symbol_strategy_returns["net_strategy_return"] - symbol_strategy_returns["passive_return"]

assert np.nanmax(np.abs(symbol_strategy_returns["gross_strategy_return"] - calc_gross)) < 1e-12, (
    "gross_strategy_return formula mismatch."
)
assert np.nanmax(np.abs(symbol_strategy_returns["cost_return"] - calc_cost)) < 1e-12, (
    "cost_return formula mismatch."
)
assert np.nanmax(np.abs(symbol_strategy_returns["net_strategy_return"] - calc_net)) < 1e-12, (
    "net_strategy_return formula mismatch."
)
assert np.nanmax(np.abs(symbol_strategy_returns["net_map_vs_passive"] - calc_map)) < 1e-12, (
    "net_map_vs_passive formula mismatch."
)

# Delayed execution check: signal_date must be strictly before execution date.
assert (symbol_strategy_returns["signal_date"] < symbol_strategy_returns["date"]).all(), (
    "Delayed execution alignment failed: signal_date should be before date."
)

# Each strategy row must use an executable forward return.
assert symbol_strategy_returns["fwd_tradable_return_flag"].all(), (
    "Some strategy rows do not have tradable forward returns."
)
assert symbol_strategy_returns["fwd_ret_1d"].notna().all(), (
    "Some strategy rows have missing fwd_ret_1d."
)

# Vol bucket legality and RV alignment.
allowed_buckets = {"Low", "Mid", "High"}
assert set(symbol_strategy_returns["vol_bucket"].dropna().unique()).issubset(allowed_buckets), (
    "Invalid vol bucket labels."
)
assert symbol_strategy_returns["vol_bucket"].notna().all(), "Strategy rows have missing vol_bucket."
assert symbol_strategy_returns["rv_60"].notna().all(), "Strategy rows have missing delayed RV60."
assert not (
    feature_panel["vol_bucket_signal"].notna()
    & feature_panel["rv_60_signal"].isna()
).any(), "Feature panel has bucket without RV60."
assert not hml_daily_returns["hml_return"].isna().any(), "HML daily returns contain NaN."

# Position must match delayed signal_close versus delayed MA.
_bad_long_flat = symbol_strategy_returns[
    (symbol_strategy_returns["version"] == "long_flat")
    & (
        ((symbol_strategy_returns["signal_close"] > symbol_strategy_returns["ma_value"]) & (symbol_strategy_returns["position"] != 1.0))
        | ((symbol_strategy_returns["signal_close"] <= symbol_strategy_returns["ma_value"]) & (symbol_strategy_returns["position"] != 0.0))
    )
]
_bad_long_short = symbol_strategy_returns[
    (symbol_strategy_returns["version"] == "long_short")
    & (
        ((symbol_strategy_returns["signal_close"] > symbol_strategy_returns["ma_value"]) & (symbol_strategy_returns["position"] != 1.0))
        | ((symbol_strategy_returns["signal_close"] <= symbol_strategy_returns["ma_value"]) & (symbol_strategy_returns["position"] != -1.0))
    )
]
assert len(_bad_long_flat) == 0, "long_flat position formula mismatch."
assert len(_bad_long_short) == 0, "long_short position formula mismatch."

# Delayed alignment spot-check against feature panel.
for w in MA_WINDOWS:
    for version in ["long_flat", "long_short"]:
        y = symbol_strategy_returns[
            (symbol_strategy_returns["ma_window"] == w)
            & (symbol_strategy_returns["version"] == version)
        ].copy()
        assert len(y) > 0, f"No rows for MA{w} {version}."

print("Rows:", len(symbol_strategy_returns))
print("Bucket rows:", len(bucket_daily_returns))
print("HML rows:", len(hml_daily_returns))
print("Date range:", symbol_strategy_returns["date"].min().date(), "->", symbol_strategy_returns["date"].max().date())
print("Notebook 02 final hard self-check: PASSED")

print("Notebook 02 finished.")
print("Main output for Notebook 03:")
print(SYMBOL_RETURNS_PARQUET)
print(SYMBOL_RETURNS_CSV)
print(BUCKET_RETURNS_PARQUET)
print(BUCKET_RETURNS_CSV)


RUN_TIME: 2026-06-19 20:46:38
INPUT_PARQUET: /home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.parquet
INPUT_CSV: /home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.csv
OUTPUT_DIR: /home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly
MA_WINDOWS: [10, 20, 60, 120]
RV_WINDOW: 60
COST_BPS: 5.0
EXECUTION_MODE: delayed_one_trading_day
Loading parquet: /home/zilinm2/futures_anomaly_project/notebook01_base_clean_panel/notebook01_base_clean_panel.parquet
panel shape: (58591, 67)
symbols: 27
date range: 2016-01-05 -> 2026-06-18


,date,trading_day,datetime,symbol,tq_symbol,exchange,product,sector,data_source,contract_mode,...,tradable_row_flag,valid_return_flag,tradable_return_flag,fwd_valid_return_flag,fwd_tradable_return_flag,extreme_return_10pct_flag,extreme_return_15pct_flag,extreme_return_20pct_flag,robust_return_outlier_flag,large_calendar_gap_flag
0,2016-01-05,2016-01-05,1.451923e+18,AG,KQ.m@SHFE.ag,SHFE,ag,precious_metal,TQSDK,MAIN_CONTINUOUS,...,True,False,False,True,True,False,False,False,False,False
1,2016-01-06,2016-01-06,1.452010e+18,AG,KQ.m@SHFE.ag,SHFE,ag,precious_metal,TQSDK,MAIN_CONTINUOUS,...,True,True,True,True,True,False,False,False,False,False
2,2016-01-07,2016-01-07,1.452096e+18,AG,KQ.m@SHFE.ag,SHFE,ag,precious_metal,TQSDK,MAIN_CONTINUOUS,...,True,True,True,True,True,False,False,False,False,False
3,2016-01-08,2016-01-08,1.452182e+18,AG,KQ.m@SHFE.ag,SHFE,ag,precious_metal,TQSDK,MAIN_CONTINUOUS,...,True,True,True,True,True,False,False,False,False,False
4,2016-01-11,2016-01-11,1.452442e+18,AG,KQ.m@SHFE.ag,SHFE,ag,precious_metal,TQSDK,MAIN_CONTINUOUS,...,True,True,True,True,True,False,False,False,False,False


Saved: /home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly/tables/notebook02_sample_coverage.csv


,date,n_symbols_total,n_rv60_signal,n_vol_bucket_signal,n_exec_vol_bucket
0,2016-01-05,20,0,0,0
1,2016-01-06,20,0,0,0
2,2016-01-07,20,0,0,0
3,2016-01-08,20,0,0,0
4,2016-01-11,20,0,0,0


,date,symbol,sector,close,ret_1d,fwd_ret_1d,fwd_log_ret_1d,tradable_return_flag,fwd_tradable_return_flag,rv_60_signal,...,exec_vol_bucket,exec_rv_60,exec_ma_10,signal_close_10,exec_ma_20,signal_close_20,exec_ma_60,signal_close_60,exec_ma_120,signal_close_120
0,2016-01-05,AG,precious_metal,3313.0,NaN,0.000302,0.000302,False,True,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-01-06,AG,precious_metal,3314.0,0.000302,0.012975,0.012892,True,True,NaN,...,<NA>,NaN,NaN,3313.0,NaN,3313.0,NaN,3313.0,NaN,3313.0
2,2016-01-07,AG,precious_metal,3357.0,0.012975,-0.000298,-0.000298,True,True,NaN,...,<NA>,NaN,NaN,3314.0,NaN,3314.0,NaN,3314.0,NaN,3314.0
3,2016-01-08,AG,precious_metal,3356.0,-0.000298,-0.002086,-0.002088,True,True,NaN,...,<NA>,NaN,NaN,3357.0,NaN,3357.0,NaN,3357.0,NaN,3357.0
4,2016-01-11,AG,precious_metal,3349.0,-0.002086,-0.018214,-0.018382,True,True,NaN,...,<NA>,NaN,NaN,3356.0,NaN,3356.0,NaN,3356.0,NaN,3356.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2016-04-27,AG,precious_metal,3774.0,0.028058,-0.004505,-0.004515,True,True,0.013341,...,Mid,0.012955,3615.2,3671.0,3508.95,3671.0,3444.500000,3671.0,NaN,3671.0
76,2016-04-28,AG,precious_metal,3757.0,-0.004505,0.019697,0.019505,True,True,0.013368,...,Mid,0.013341,3644.3,3774.0,3528.00,3774.0,3451.400000,3774.0,NaN,3774.0
77,2016-04-29,AG,precious_metal,3831.0,0.019697,-0.001044,-0.001045,True,True,0.013540,...,Mid,0.013368,3671.5,3757.0,3544.65,3757.0,3457.933333,3757.0,NaN,3757.0
78,2016-05-03,AG,precious_metal,3827.0,-0.001044,-0.021427,-0.021660,True,True,0.013507,...,Mid,0.013540,3705.4,3831.0,3565.90,3831.0,3465.916667,3831.0,NaN,3831.0


...

Saved: /home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly/tables/notebook02_monotonic_flags.csv


,ma_window,version,High,Low,Mid,hml_mean_daily,hml_t_stat,hml_sharpe,n_obs,monotonic_flag
0,10,long_flat,0.000163,0.000138,0.000095,0.000026,0.146967,0.046886,2476,PARTIAL
1,10,long_short,-0.000062,0.000019,-0.000124,-0.000080,-0.352990,-0.112613,2476,REVERSED
2,20,long_flat,0.000282,0.000111,0.000144,0.000171,0.964578,0.307725,2476,PARTIAL
3,20,long_short,0.000175,-0.000034,-0.000027,0.000209,0.910189,0.290373,2476,PARTIAL
4,60,long_flat,0.000235,0.000165,0.000190,0.000070,0.384117,0.122543,2476,PARTIAL
5,60,long_short,0.000081,0.000074,0.000066,0.000007,0.030511,0.009734,2476,PARTIAL
6,120,long_flat,0.000029,0.000165,0.000064,-0.000137,-0.766591,-0.247529,2417,REVERSED
7,120,long_short,-0.000304,0.000102,-0.000160,-0.000406,-1.766526,-0.570403,2417,REVERSED


Saved: /home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly/tables/notebook02_run_config.csv


,key,value
0,run_time,2026-06-19 20:46:38
1,input_parquet,/home/zilinm2/futures_anomaly_project/notebook...
2,input_csv,/home/zilinm2/futures_anomaly_project/notebook...
3,output_dir,/home/zilinm2/futures_anomaly_project/notebook...
4,execution_mode,delayed_one_trading_day
5,execution_note,Version A: position on date t equals signal po...
6,return_window,fwd_ret_1d[t] = close[t+1] / close[t] - 1
7,rv_window,60
8,ma_windows,"10,20,60,120"
9,min_cross_section_symbols,9


Notebook 02 final hard self-check
Rows: 452150
Bucket rows: 59070
HML rows: 118140
Date range: 2016-04-07 -> 2026-06-17
Notebook 02 final hard self-check: PASSED
Notebook 02 finished.
Main output for Notebook 03:
/home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly/strategy_returns/notebook02_symbol_ma_strategy_returns.parquet
/home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly/strategy_returns/notebook02_symbol_ma_strategy_returns.csv
/home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly/strategy_returns/notebook02_bucket_daily_returns.parquet
/home/zilinm2/futures_anomaly_project/notebook02_vol_sorted_technical_anomaly/strategy_returns/notebook02_bucket_daily_returns.csv
